# Radar DSP Pipeline Walkthrough

This notebook is a reproducible walkthrough of the `radar-dsp-pipeline`
repository.

The pipeline demonstrates how practical DSP decisions affect radar-relevant
measurement, detection, and numerical implementation behavior:

1. A signal can be buried in noise.
2. FFT processing can expose a spectral or Doppler-like peak.
3. Peak visibility must be evaluated relative to a noise floor.
4. Filtering changes spectral content, noise bandwidth, and detectability.
5. Windowing changes leakage, coherent gain, scalloping loss, and equivalent
   noise bandwidth.
6. Doppler-like detection requires threshold calibration and empirical Pd/Pfa
   estimation.
7. System-level DSP choices produce measurable Pd, Pfa, SNR, and computational
   trade-offs.
8. IIR pole location, arithmetic precision, and implementation structure affect
   stability and numerical robustness.

The reusable numerical implementation lives in `src/`; the executable pipeline
stages live in `scripts/`.

This notebook orchestrates the stages, displays generated artifacts, and
provides a concise engineering interpretation of the results.

## 0. Repository setup

This cell resolves the repository root, configures imports, and verifies the expected project structure.

In [ ]:
from __future__ import annotations

import os
import sys
import subprocess
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

# Resolve repository root from the notebook location.
# Expected notebook path:
#   radar-dsp-pipeline/notebooks/radar_dsp_pipeline_walkthrough.ipynb
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXPECTED_PATHS = [
    REPO_ROOT / "src" / "signals.py",
    REPO_ROOT / "src" / "fft_tools.py",
    REPO_ROOT / "src" / "filters.py",
    REPO_ROOT / "src" / "windows.py",
    REPO_ROOT / "src" / "detection.py",
    REPO_ROOT / "scripts" / "01_signals_noise_fft.py",
    REPO_ROOT / "scripts" / "02_fir_iir_filters.py",
    REPO_ROOT / "scripts" / "03_windowing_leakage.py",
    REPO_ROOT / "scripts" / "04_detection_doppler.py",
    REPO_ROOT / "scripts" / "05_system_tradeoffs.py",
]

missing = [p for p in EXPECTED_PATHS if not p.exists()]
if missing:
    raise FileNotFoundError("Missing expected repository files:\n" + "\n".join(str(p) for p in missing))

FIGURES_DIR = REPO_ROOT / "figures" / "generated_plots"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"[OK] Repository root: {REPO_ROOT}")
print(f"[OK] Figures directory: {FIGURES_DIR.relative_to(REPO_ROOT)}")

## 1. Utility functions

The notebook runs scripts exactly as pipeline stages.  
Each stage is executed from the repository root, so all paths remain repo-relative and reproducible.

In [ ]:
def run_stage(script_name: str, *args: str) -> None:
    """
    Run a pipeline stage from the repository root.

    Parameters
    ----------
    script_name:
        Filename inside the scripts/ directory.
    *args:
        Optional command-line arguments passed to the script.
    """
    script_path = REPO_ROOT / "scripts" / script_name
    cmd = [sys.executable, str(script_path), *args]

    print("[RUN]", " ".join(cmd))

    completed = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    print(completed.stdout)

    if completed.returncode != 0:
        raise RuntimeError(f"Stage failed with exit code {completed.returncode}: {script_name}")


def show_artifact(relative_path: str, width: int = 950) -> None:
    """
    Display a generated image artifact from the repository.

    Parameters
    ----------
    relative_path:
        Repository-relative artifact path.
    width:
        Display width in pixels.
    """
    path = REPO_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)

    print(f"[ARTIFACT] {relative_path}")
    display(Image(filename=str(path), width=width))


def read_csv_artifact(relative_path: str) -> pd.DataFrame:
    """
    Load a generated CSV artifact from the repository.
    """
    path = REPO_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

## 2. Stage 01 — Signal buried in noise, FFT visibility, and detectability

Stage 01 demonstrates the first radar-DSP idea: a weak sinusoidal component may be hard to see in time domain, but a narrowband peak can emerge in the frequency domain.

The script generates several SNR cases, computes the FFT, detects the spectral peak, estimates the noise floor, and reports peak-to-noise-floor detectability.

In [ ]:
run_stage("01_signals_noise_fft.py")

show_artifact("figures/generated_plots/01_multi_snr_spectrum.png")
show_artifact("figures/generated_plots/01_detectability_vs_snr.png")

## 3. Stage 02 — FIR/IIR filtering and detectability

Stage 02 compares low-pass FIR and IIR filters.

The point is not that filtering is automatically good. The point is that filter design changes spectral content, noise bandwidth, and therefore the measured visibility of a signal component.

In [ ]:
run_stage("02_fir_iir_filters.py")

show_artifact("figures/generated_plots/02_filter_response_fir_vs_iir.png")
show_artifact("figures/generated_plots/02_filtered_signal_spectrum.png")
show_artifact("figures/generated_plots/02_filter_detectability_summary.png")

## 4. Stage 03 — Windowing, leakage, coherent gain, and ENBW

Stage 03 shows why windowing is not cosmetic.

Windows reduce leakage but modify coherent gain and equivalent noise bandwidth. In radar/Doppler processing, that trade-off matters because it affects both amplitude interpretation and noise behavior.

In [ ]:
run_stage("03_windowing_leakage.py")

show_artifact("figures/generated_plots/03_window_leakage_comparison.png")
show_artifact("figures/generated_plots/03_window_metrics_summary.png")
show_artifact("figures/generated_plots/03_detectability_window_tradeoff.png")

## 5. Stage 04 — Doppler-like peak detection and empirical Pd/Pfa

Stage 04 moves from visibility to detection.

A peak can be visible, but the detector still needs a threshold. This stage estimates a threshold from noise-only trials, then measures empirical probability of detection and false-alarm behavior.

In [ ]:
run_stage("04_detection_doppler.py")

show_artifact("figures/generated_plots/04_doppler_peak_detection.png")
show_artifact("figures/generated_plots/04_pd_pfa_basic.png")

## 6. Stage 05 — System-level DSP trade-offs

Stage 05 is the engineering summary.

It compares processing chains using:

- fixed-threshold detection
- CA-CFAR detection
- empirical Pd vs SNR
- empirical Pfa control
- interpolated SNR required to reach target Pd

This is the part of the repository that connects individual DSP choices to system-level detection behavior.

In [ ]:
run_stage("05_system_tradeoffs.py")

show_artifact("figures/generated_plots/05_fixed_threshold_pd_vs_snr.png")
show_artifact("figures/generated_plots/05_ca_cfar_pd_vs_snr.png")
show_artifact("figures/generated_plots/05_false_alarm_control.png")
show_artifact("figures/generated_plots/05_snr_requirement_summary.png")

## 7. Stage 05 numerical summary

The CSV output is the compact engineering summary.  
It is useful for reviewing detector behavior without inspecting every plot.

In [ ]:
df_metrics = read_csv_artifact("figures/generated_plots/05_pipeline_metrics.csv")
display(df_metrics)

## 8. Stage 06 — IIR stability and numerical robustness

Stage 06 demonstrates that theoretical pole stability and practical numerical
robustness are related but not identical.

It compares:

- stable poles inside the unit circle
- poles close to the unit circle
- unstable poles outside the unit circle
- direct-form filtering
- second-order-section (SOS) filtering
- float32 and float64 arithmetic

The unit-impulse response reveals whether each recursive implementation decays,
remains numerically bounded, overflows, or diverges.

The key engineering result is that a theoretically stable high-order filter can
still fail numerically when implemented in direct form with limited precision.
SOS implementation reduces coefficient sensitivity and internal numerical
growth, but it cannot make a theoretically unstable filter stable.

**Plot interpretation note:** Curves that spike and then collapse to the
plotting floor correspond to overflow or non-finite sample sanitation, not to
physical decay of the filter response.

In [ ]:
run_stage("06_iir_stability_demo.py")

show_artifact(
    "figures/generated_plots/06_iir_stability_impulse_response.png"
)

## 9. Interpretation

The pipeline demonstrates a progression from signal representation to
system-level detection behavior and numerical implementation robustness.

Key conclusions from the default run:

- FFT analysis can expose a weak narrowband component that is not obvious in
  the time domain.
- Detectability is not the same as detection; thresholding changes the problem
  from spectral visibility to a controlled decision process.
- Filters and windows alter signal amplitude, noise bandwidth, leakage, and
  spectral resolution in measurable ways.
- Fixed thresholds can be calibrated from Monte Carlo H0 trials to target an
  empirical probability of false alarm.
- CA-CFAR must be evaluated in linear power and at a defined cell under test.
- Required SNR at a target Pd provides a compact engineering metric for
  comparing complete DSP chains.
- IIR stability depends first on pole location: poles inside the unit circle
  are theoretically stable, while poles outside it produce divergent impulse
  responses.
- Theoretical stability does not guarantee numerical robustness. A
  near-unit-circle, high-order filter can overflow in direct form with float32
  arithmetic even though its poles remain inside the unit circle.
- Second-order sections reduce coefficient sensitivity and internal numerical
  growth compared with a single high-order direct-form implementation.
- SOS implementation cannot stabilize a filter whose poles are outside the
  unit circle; it only delays or reduces numerical overflow during a finite
  observation.

From an engineering perspective, the repository connects four levels of
reasoning:

1. Signal representation and spectral measurement.
2. DSP choices such as filtering and windowing.
3. Statistical detection performance through Pd and Pfa.
4. Practical numerical implementation through precision and filter structure.

Overall, the repository provides a reproducible DSP analysis pipeline with
explicit assumptions, validated numerical primitives, generated artifacts,
measurable detection consequences, and implementation-level trade-offs
relevant to radar processing software.

## 10. Reproducibility checklist

A clean run should regenerate all expected artifacts under:

    figures/generated_plots/

Expected Stage 01 artifacts:

    01_multi_snr_spectrum.png
    01_detectability_vs_snr.png

Expected Stage 02 artifacts:

    02_filter_response_fir_vs_iir.png
    02_filtered_signal_spectrum.png
    02_filter_detectability_summary.png

Expected Stage 03 artifacts:

    03_window_leakage_comparison.png
    03_window_metrics_summary.png
    03_detectability_window_tradeoff.png

Expected Stage 04 artifacts:

    04_doppler_peak_detection.png
    04_pd_pfa_basic.png

Expected Stage 05 artifacts:

    05_fixed_threshold_pd_vs_snr.png
    05_ca_cfar_pd_vs_snr.png
    05_false_alarm_control.png
    05_snr_requirement_summary.png
    05_pipeline_metrics.csv

Expected Stage 06 artifact:

    06_iir_stability_impulse_response.png

A reproducible run is complete only when every stage executes without errors
and every artifact listed above is regenerated from the repository code.